In [1]:
import sys
sys.path.append('..')
import pandas as pd
from src.db import read_sql
from src.cleaning import load_clean_orders
from src.features import get_first_orders, build_target, add_features, apply_cutoff

df = load_clean_orders()
last_date = df['order_purchase_timestamp'].max()
first = get_first_orders(df)
target = build_target(df, window_days=180)
pf = read_sql('SELECT * FROM vw_order_product_features')

feat = add_features(first, pf)
feat = feat.merge(target, left_on='customer_unique_id', right_index=True)
feat = apply_cutoff(feat, last_date, window_days=180)

print('customers:', len(feat), ' eligible share:', round(feat['eligible'].mean(), 3))
print('repeat rate among eligible:', round(feat.loc[feat.eligible == 1, 'repeat_180d'].mean(), 4))
feat.to_csv('../data/processed/model_dataset.csv', index=False)
feat.head()

customers: 93350  eligible share: 0.599
repeat rate among eligible: 0.0312


,customer_unique_id,order_purchase_timestamp,items_value,n_items,freight_total,freight_ratio,delivery_days,delivery_delay_days,is_late,review_score,...,purchase_month,purchase_dow,is_weekend,is_holiday_season,main_category,main_payment_type,customer_state,delay_bucket,repeat_180d,eligible
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,129.90,1,12.00,0.092379,6,-5,0,5.0,...,5,3,0,0,bed_bath_table,credit_card,SP,on_time,0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,18.90,1,8.29,0.438624,3,-5,0,4.0,...,5,0,0,0,health_beauty,credit_card,SP,on_time,0,0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,69.00,1,17.22,0.249565,26,-2,0,3.0,...,3,4,0,0,stationery,credit_card,SC,on_time,0,1
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,25.99,1,17.63,0.678338,20,-12,0,4.0,...,10,3,0,0,telephony,credit_card,PA,on_time,0,1
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,180.00,1,16.89,0.093833,13,-8,0,5.0,...,11,1,0,1,telephony,credit_card,SP,on_time,0,1
